<a href="https://colab.research.google.com/github/TCU-DCDA/WRIT20833_2026/blob/main/notebooks/codeAlongs/WRIT20833_Read_First_Then_Reveal_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WRIT 20833 — Read First, Then Reveal

**When Coding Meets Culture: Developing Data-Driven Opinions**

Make an editable copy of this worksheet by going to **File > Save a copy in Drive**

**New to Colab, or submitting?** See [Working in Colab](https://tcu-dcda.github.io/WRIT20833_2026/guides/working-in-colab.html).

*This is a separate notebook on purpose: your work in the Sentiment with VADER notebook stays where it is.*

**How this notebook works.** One code cell is blank on purpose: in Step 2 you type the class's
stance labels into it. Every other cell is ready to run. If a cell gives you an error, read the message.
Errors are part of the work here, not something to hide.

## Why we're doing this

- **From the lecture:** close reading and distant reading answer different questions. VADER reads every
  comment at once and hands back a number. You read a few comments slowly and judge what the writer means.
  This notebook puts the two side by side.
- **Where you'll use it:** HW3 **B3**. This notebook runs B3's two cells, line for line, on real comments,
  so the steps are familiar before you do them on your own data.
- **The question to keep asking:** what did VADER's number flatten, and could you have read the writer's
  side off their tone?

The order matters. If you see VADER's score first, it nudges your reading ("+0.99, so it must be
friendly"). So we **read first and label, then reveal** the scores.

# Setup

The same tools as the Sentiment with VADER notebook, with the same names HW3 uses: `get_sentiment` turns a
comment into its compound score, and `label_sentiment` turns a score into "positive", "negative", or
"neutral".

In [ ]:
!pip install -q vaderSentiment==3.3.2

In [ ]:
# PLUMBING: the tools from the VADER notebook (Parts 2 and 4)
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    return analyzer.polarity_scores(text)["compound"]

def label_sentiment(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    return "neutral"

print("VADER ready.")

## The comments

They come from a second video on the same Texas law: a Texas House exchange about the bill, posted by
KXAN and collected with the course's YouTube Comment Collector (1,509 comments; see the course data page).
We keep only the **top-level** comments. Replies are arguments with other commenters, and their stance is
about the person they're answering, which makes them harder to label.

We use the names HW3 uses, `df` and `text_column`, so the B3 cells below run exactly as they will on your
own file.

In [ ]:
# PLUMBING: load the KXAN comments, keep the top-level ones, and score them
url = "https://tcu-dcda.github.io/WRIT20833_2026/data/ten_commandments_kxan/tc_kxan_youtube_comments.csv"
kxan = pd.read_csv(url, keep_default_na=False)
df = kxan[kxan["is_reply"] == False].copy()   # top-level comments only; .copy() makes it a table of its
                                               # own, so adding columns below doesn't set off a warning
text_column = "comment_clean"

df["sentiment"] = df[text_column].apply(get_sentiment)
df["label"] = df["sentiment"].apply(label_sentiment)
print(len(df), "top-level comments scored.")

# Step 1: Read the Extremes, with the Scores Hidden

This is HW3's B3 part 1, line for line. It picks the 5 comments VADER scored highest and the 5 it scored
lowest, then prints only their text, in row order, so you can't tell which group a comment came from.

Some are long. That's a pattern worth noticing: the highest and lowest scores often go to the longest
comments, because a long comment piles up more scored words.

**What B3 part 1 does, step by step.** The numbers match the `# 1` to `# 6` tags in the code cell.

1. `min(5, len(df) // 2)`: 5 comments from each end, or half your rows if you have fewer than 10, so
   both groups are the same size. (`//` divides and drops the remainder: `7 // 2` is 3.)
2. `.nlargest(how_many, "sentiment")`: keeps the rows with the highest scores.
3. `.drop(highest.index)`: sets those rows aside first, so `.nsmallest` can't pick any of them again.
   Without it, a small or one-sided dataset could put the same comment in both groups.
4. `.assign(group=...)` adds a column saying which group each row came from; `pd.concat([...])` stacks
   the two small tables into one.
5. `.sort_index()` puts the rows in row-number order, which mixes the two groups together;
   `.iterrows()` hands you the table one row at a time, as a pair: the row's number and the row itself.
6. `textwrap.fill(...)` breaks a long comment into lines of at most 90 characters, indented 4 spaces,
   so you can read it without scrolling sideways. (`import textwrap` at the top loads that tool; it
   comes with Python.)

In [ ]:
# B3 — part 1: read the extremes (scores hidden). Steps 1-6 are explained in the text above.
# Run this cell, then read the comments it prints.
import textwrap
how_many = min(5, len(df) // 2)                                     # 1
highest = df.nlargest(how_many, "sentiment")                        # 2
lowest = df.drop(highest.index).nsmallest(how_many, "sentiment")    # 3
extremes = pd.concat([highest.assign(group="highest"),
                      lowest.assign(group="lowest")])               # 4
print(len(highest), "highest-scoring and", len(lowest), "lowest-scoring comments, in row order:")
print()
for row_number, row in extremes.sort_index().iterrows():            # 5
    print(f"[{row_number}]")
    print(textwrap.fill(row[text_column], width=90,
                        initial_indent="    ", subsequent_indent="    "))   # 6
    print()

# Step 2: Label Each Comment's Stance

The sides here: does the writer **support** or **oppose** putting the Ten Commandments in Texas
classrooms? Use **unclear** when you can't tell, and **n/a** when the comment isn't about that question at
all. Decide from the words alone. Agree on a label for each before anyone runs Step 3.

You type the labels into a dictionary, and pandas attaches them to the table.

**The structure:**
```python
my_stance = {
    row_number: "support",
    another_row_number: "oppose",
}
extremes["stance"] = extremes.index.map(my_stance)
```
- Each **key** is a row number from Step 1's printout (the number in square brackets), typed without
  quotes. Each **value** is your label, in quotes.
- `extremes.index` is the table's row numbers. `.map(my_stance)` looks each one up in your dictionary
  and gives back your label, like `my_stance[row_number]` done for every row at once.
- A row you haven't labeled yet shows `NaN` (pandas' mark for an empty cell), so you can see which
  ones are left.

In [ ]:
# type along — one entry per [row number] from Step 1: "support", "oppose", "unclear", or "n/a"
my_stance = {

}

# Step 3: Reveal

Only now do VADER's scores appear beside your labels. This is HW3's B3 part 2.

In [ ]:
# B3 — part 2: attach your labels, then reveal the scores
extremes["stance"] = extremes.index.map(my_stance)
extremes[["group", "sentiment", "label", "stance"]]   # the "highest" group first, then the "lowest"

**Talk it through.** These are the questions HW3's B3 reflection asks about *your* data:
- In the **highest** group, how many did you label "support"? In the **lowest** group?
- If positive tone meant support, what would you expect to see? What did you see?
- Before the reveal, did any comment seem to you to have a different tone than VADER gave it?
- Which labels were hard to agree on, and why? That disagreement is data too.

# Sneak Preview: Where This Is Going

**In HW3 B3** you'll run these same two cells on your own file. The only differences: **you** decide what
your comments are arguing about, and the labels are yours alone.

**In your capstone**, this is the move that makes a claim trustworthy: before you report what a tool found,
read some of what it scored and say where you agree with it and where you don't.

# Playground

Three places to start:
- Change `5` to `8` in `min(5, len(df) // 2)` and rerun Steps 1–3. Do more comments change the pattern?
- Pick one comment from Step 1 and score just its first sentence with `get_sentiment`. How much of the
  high score came from length?
- Find a short comment in `df` that you think VADER gets badly wrong, and say why.

In [ ]:
# use this space to experiment!